# Assignment 2
#### Casey Elizabeth Gilbert

In [ ]:
# import a bunch of stuff
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from matplotlib import rc
import time
import random
import math
from collections import Counter

## Question 1

#### Question 1a

In [ ]:
def grow_barabasi_albert_net(N, m, seed=None, debug=False):
    """
    This function creates a scale-free network using barabasi-albert model.

    It loops through and adds the proper amount of nodes (specified as param).
    For each node, it connects via m links to existing nodes. 

    NOTES:
    Adds links with preferential attachment, the preferential attachment comes from choosing a random edge.
    So a node with twice as many edges is twice as likely to have an edge chosen.
    Because degree = edge count * 2, th probability for each node is then halved
    This is taken care of when we randomly select one of the edge's nodes.
    
    This doesn't allow self loops, and isn't exactly LCD, but it is computationally cleaner,
    and I think should create scale-free networks.

    @param N (int): The number of nodes
    @param m (int): Number of edges formed for each new node
    @param seed (int?): The seed for the network (optional)
    @param debug (bool): Whether or not you want print statements

    @return the scale-free network
    @return an array of degree sequences at N^x
    """

    # Create the graph
    degree_sequences = []
    simple_graph = nx.complete_graph(4)
    start_graph = nx.MultiGraph(simple_graph) # multilinks are allowed

    # Set our seed
    random.seed(seed)

    # Our time counter
    # Start with 4 because the network always starts with 4 nodes
    t = 4

    # get our order of magnitude so we know what to print
    o_of_m = int(math.log10(N))

    degree_sequences_vals = []
    if (o_of_m > 1):
        for i in range(2,o_of_m + 1):
            degree_sequences_vals.append(10**i)
    if debug: print(degree_sequences_vals)
    while t < N:
        t = t + 1

        # add node
        start_graph.add_node(t)

        # we don't want to accidentally select the new node's edge,
        # so we get the edge list ahead of time
        edge_list = list(start_graph.edges) 

        # Now add the m edges
        # Note that this doesn't allow self-links
        # But I don't think that's a requirement for BA; according to the book
        # BA leaves many mathematical details open
        for i in range(0,m):
            # get random edge
            chosen_edge = edge_list[random.randint(0,len(edge_list)-1)]
            if debug: print(f"chosen_edge: {chosen_edge}")

            #randomly choose one edge of that link
            chosen_node = chosen_edge[random.randint(0,1)]
            if debug: print(f"Chosen node: {chosen_node}")

            # attach the new edge 
            start_graph.add_edge(t, chosen_node)

        # we want the degree sequence of certain orders of magnitude
        if t in degree_sequences_vals:
            degree_seq = np.array(list(dict(start_graph.degree()).values()))
            degree_sequences.append(degree_seq)


    return start_graph, degree_sequences
    

In [ ]:
def grow_barabasi_albert_net_later_option(N, m, seed=None, debug=False):
    """
    This function creates a scale-free network using barabasi-albert model.

    It loops through and adds the proper amount of nodes (specified as param).
    For each node, it connects via m links to existing nodes. 

    NOTES:
    This is a version I added because it preferentially adds links including the links 
    added as m. Which originally I did not do. And am still not clear if BA requires this. 
    But the below analysis was done with only using preferential attachement for nodes that
    are not new. 

    @param N (int): The number of nodes
    @param m (int): Number of edges formed for each new node
    @param seed (int?): The seed for the network (optional)
    @param debug (bool): Whether or not you want print statements

    @return the scale-free network
    @return an array of degree sequences at N^x
    """

    # Create the graph
    degree_sequences = []
    simple_graph = nx.complete_graph(4)
    start_graph = nx.MultiGraph(simple_graph) # multilinks are allowed

    # Set our seed
    random.seed(seed)

    # Our time counter
    # Start with 4 because the network always starts with 4 nodes
    t = 4
    edge_list = list(start_graph.edges) 

    # get our order of magnitude so we know what to print
    o_of_m = int(math.log10(N))

    degree_sequences_vals = []
    if (o_of_m > 1):
        for i in range(2,o_of_m + 1):
            degree_sequences_vals.append(10**i)
    if debug: print(degree_sequences_vals)
    while t < N:
        t = t + 1

        # add node
        start_graph.add_node(t)

        edge_list = list(start_graph.edges) 

        # Now add the m edges
        # Note that this doesn't allow self-links
        # But I don't think that's a requirement for BA; according to the book
        # BA leaves many mathematical details open
        for i in range(0,m):
            # get random edge
            chosen_edge = edge_list[random.randint(0,len(edge_list)-1)]
            if debug: print(f"chosen_edge: {chosen_edge}")

            #randomly choose one edge of that link
            chosen_node = chosen_edge[random.randint(0,1)]
            if debug: print(f"Chosen node: {chosen_node}")

            # attach the new edge 
            start_graph.add_edge(t, chosen_node)
            edge_list.append((t, chosen_node))

        # we want the degree sequence of certain orders of magnitude
        if t in degree_sequences_vals:
            degree_seq = np.array(list(dict(start_graph.degree()).values()))
            degree_sequences.append(degree_seq)


    return start_graph, degree_sequences

In [ ]:
N = 10**4
m = 4
# Just test it with smaller numbers to see what's up
G, degree_dists = grow_barabasi_albert_net(124,2)

plt.figure(figsize=(4, 4))
nx.draw(G, with_labels=True, node_color='lightblue', edge_color='gray', node_size=100, font_size=10)
plt.title("B-A Graph")
plt.show()

In [ ]:
N = 10**4
m = 4
# Just test it with smaller numbers to see what's up
G, degree_dists = grow_barabasi_albert_net(N,m)

In [ ]:
degree_dists

In [ ]:
fig, ax = plt.subplots(1,len(degree_dists),figsize=(14,4),dpi=100)

for i in range(0, len(degree_dists)):
    # get nice binned distributions 
    deg_vals = degree_dists[i]
    deg_bins = np.logspace(0, np.log10(np.max(deg_vals)+1), num=40)
    probs, _ = np.histogram(deg_vals, bins=deg_bins, density=True)
    final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

    # plot that puppy
    ax[i].loglog(final_bins, probs, marker='o',lw=0)
    ax[i].set_xlabel(r"$k$", fontsize='medium')
    ax[i].set_ylabel(r"$P(k)$", fontsize='medium')
    ax[i].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
    ax[i].set_title(f'Degree Distribution at {10**(i+2)}', fontsize='large')

plt.show()

#### Question 1b

In [ ]:
import powerlaw

for i,dist in enumerate(degree_dists):
    print(f"Dist: {10**(i+2)}")
    fit = powerlaw.Fit(dist)

    print(fit.power_law.alpha)
    print(fit.power_law.xmin)

In [ ]:
fig, ax = plt.subplots(1,len(degree_dists),figsize=(14,4),dpi=100)

for i,dist in enumerate(degree_dists):
    # fit the distribution
    fit = powerlaw.Fit(dist)

    # get nice binned distributions 
    deg_vals = degree_dists[i]
    deg_bins = np.logspace(0, np.log10(np.max(deg_vals)+1), num=40)
    probs, _ = np.histogram(deg_vals, bins=deg_bins, density=True)
    final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

    # plot that puppy
    ax[i].loglog(final_bins, probs, marker='o',lw=0)
    ax[i].set_xlabel(r"$k$", fontsize='medium')
    ax[i].set_ylabel(r"$P(k)$", fontsize='medium')
    ax[i].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
    ax[i].legend([fit.power_law.alpha, fit.power_law.xmin], ['γ', 'k-min'], fontsize='small')
    ax[i].set_title(f'Degree Distribution at {10**(i+2)}', fontsize='large')

    box_style = dict(facecolor='white', edgecolor='lightgrey', boxstyle='square,pad=0.5', alpha=0.7)
    fit_info = f"γ: {round(fit.power_law.alpha,2)} \nk-min: {round(fit.power_law.xmin,2)}"
    ax[i].text(0.05, 0.18, fit_info, verticalalignment='top', transform=ax[i].transAxes, fontsize=11, bbox=box_style)


plt.savefig('images/deg_dist_graphs.pdf', bbox_inches='tight')
plt.show()

In [ ]:
for i,dist in enumerate(degree_dists):
    print(f"Dist {i}")
    dist = degree_dists[i]
    unique = list(set(dist))
    unique.sort()
    for k in unique: 
        num = 2*m * (m+1) 
        den = k*(k+1)*(k+2)
        print(f"deg: {k} and num: {num} and den: {den} and total: {num/den}")
    print("\n")

### Question 1c

In [ ]:
from tailestim import HillEstimator, KernelTypeEstimator, MomentsEstimator

def estimate_gamma(G):
    """
    This function estimates the gamma of a network's degree distribution using three different functions:
    - powerlaw estimate
    - hill estimate
    - moments estimate
    - Kernal-type estimate

    @param G (NetworkX network): The network you want to estimate

    @return a dict of the estimates, labeled "power", "hill", "moments", and "kernel"
    """

    gamma_estimates = dict()

    # Get the distribution 
    degree_seq = np.array(list(dict(G.degree()).values()))
    
    # powerlaw estimate
    fit = powerlaw.Fit(degree_seq)
    pwr_gamma = fit.power_law.alpha
    gamma_estimates["power"] = pwr_gamma

    # HillEstimator
    hill_estimator = HillEstimator()
    hill_estimator.fit(degree_seq)
    hill_result = hill_estimator.get_result()
    hill_gamma = hill_result.gamma_
    gamma_estimates["hill"] = hill_gamma

    # Moments Estimator
    moments_estimator = MomentsEstimator()
    moments_estimator.fit(degree_seq)
    moments_result = moments_estimator.get_result()
    moments_gamma = moments_result.gamma_
    gamma_estimates["moments"] = moments_gamma

    # Moments Estimator
    kernel_estimator = KernelTypeEstimator()
    kernel_estimator.fit(degree_seq)
    kernel_result = kernel_estimator.get_result()
    kernel_gamma = kernel_result.gamma_
    gamma_estimates["kernel"] = kernel_gamma

    return gamma_estimates



In [ ]:
N = 10**4
m = 4

# networks
ba_networks = []

# our arrays of estimates
power_estimates = []
hill_estimates = []
moments_estimates = []
kernel_estimates = []

for i in range(0,100): 
    # Create the network and estimate it
    random_seed = random.randint(1,20000)
    temp_G, _ = grow_barabasi_albert_net(N,m,random_seed)
    ba_networks.append(temp_G)

    # Now estimates 
    estimates = estimate_gamma(temp_G)

    # add results to our arrays
    power_estimates.append(estimates["power"])
    hill_estimates.append(estimates["hill"])
    moments_estimates.append(estimates["moments"])
    kernel_estimates.append(estimates["kernel"])

print(power_estimates)
print(hill_estimates)
print(moments_estimates)
print(kernel_estimates)

In [ ]:
# Now graph it
fig, ax = plt.subplots(1,1,figsize=(8,5),dpi=100)
plt.subplots_adjust(wspace=0.3)

#ax.plot(power_estimates, 'o', color='maroon', label='Power Estimates', alpha=0.8, mec='.1')
ax.plot(power_estimates, label='Power Estimates', color='maroon', linestyle="-", marker='s', linewidth=0.5, alpha=0.8, mec='.1')
ax.plot(hill_estimates, '+', color='cornflowerblue', linestyle="-", label='Hill Estimates', alpha=0.8, mec='.1', linewidth=0.5)
ax.plot(moments_estimates, 'o', color='green', linestyle="-", label='Moments Estimates', alpha=0.8, mec='.1', linewidth=0.5)
ax.plot(kernel_estimates, 'd', color='purple', linestyle="-", label='Kernel Estimates', alpha=0.8, mec='.1', linewidth=0.5)

ax.set_xlabel(r"Network Index", fontsize='medium')
ax.set_ylabel(r"Gamma", fontsize='medium')
ax.legend(fontsize='small')
ax.grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
ax.set_title('Gamma Estimators', fontsize='x-large')

plt.savefig('images/gamma_estimators.png', dpi=425, bbox_inches='tight')


In [ ]:
# Now graph it
fig, ax = plt.subplots(1,1,figsize=(8,5),dpi=100)
plt.subplots_adjust(wspace=0.3)

ax.plot(power_estimates, 's', label='Power Estimates', color='maroon', alpha=0.8, mec='.1')
ax.plot(hill_estimates, '+', color='cornflowerblue', label='Hill Estimates', alpha=0.8, mec='.1')
ax.plot(moments_estimates, 'o', color='green', label='Moments Estimates', alpha=0.8, mec='.1')
ax.plot(kernel_estimates, 'd', color='purple', label='Kernel Estimates', alpha=0.8, mec='.1')

ax.set_xlabel(r"Network Index", fontsize='medium')
ax.set_ylabel(r"Gamma", fontsize='medium')
ax.legend(fontsize='small')
ax.grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
ax.set_title('Gamma Estimators', fontsize='x-large')

plt.savefig('images/gamma_estimators_no-line.png', dpi=425, bbox_inches='tight')


In [ ]:
print(max(degree_seq))
print(min(degree_seq))

In [ ]:
log_likelihoods = []
p_values = []
for network in ba_networks:
    # Get the distribution 
    degree_seq = np.array(list(dict(network.degree()).values()))

    # powerlaw estimate
    fit = powerlaw.Fit(degree_seq, xmin=4, discrete=True)
    R, p = fit.distribution_compare('power_law', 'lognormal')
    log_likelihoods.append(R)
    p_values.append(p)

In [ ]:
# Now graph it
fig, ax = plt.subplots(1,1,figsize=(8,5),dpi=100)
plt.subplots_adjust(wspace=0.3)

ax.plot(log_likelihoods, 'o', label='Loglikelihood Estimates', color='cornflowerblue', alpha=0.8, mec='.1')


ax.set_xlabel(r"Network Index", fontsize='medium')
ax.set_ylabel(r"Loglikelihood", fontsize='medium')
ax.legend(fontsize='small')
ax.grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
ax.set_title('Loglikelihood Estimates', fontsize='x-large')

plt.savefig('images/log_likelihood_estimates.png', dpi=425, bbox_inches='tight')

## Question 2

#### Question 2a

In [ ]:
from scipy.io import mmread
import networkx as nx
import matplotlib.pyplot as plt

# Read in data
matrix = mmread("data/ia-reality/ia-reality.mtx")
mit_call_network = nx.Graph(matrix)

print(mit_call_network.number_of_nodes())
print(mit_call_network.number_of_edges())

hist = nx.degree_histogram(mit_call_network)
plt.figure(figsize=(8, 5))
plt.bar(range(len(hist)), hist, width=0.8, color="blue")

plt.title("Original Network Degree Distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.show()

In [ ]:
communities = nx.community.louvain_communities(mit_call_network, seed=67)

nx_node_dict = {
    node: community_id 
    for community_id, community_nodes in enumerate(communities) 
    for node in community_nodes
}

In [ ]:
import numpy as np
pos = nx.spring_layout(mit_call_network)
colors_sns = plt.cm.tab20b(np.linspace(0,1,len(np.unique(list(communities)))))
colors_nodes = [colors_sns[nx_node_dict[node]] for node in mit_call_network.nodes()]

fig, ax = plt.subplots(1,1,figsize=(12,12),dpi=100)

nx.draw_networkx(mit_call_network, pos=pos, node_color=colors_nodes, with_labels=False, ax=ax, node_size=15,width=0.4,)
plt.title("MIT Cell Phone Network Communities")

plt.savefig("images/cellphone_communities.png")

plt.show()

In [ ]:
# Get the modularity 
modularity = nx.community.quality.modularity(mit_call_network, communities)
print(f"Modularity: {modularity}")

# Get the number of communities
community_number = len(communities)
print(f"Community count: {community_number}")

# Node dictionary
print(nx_node_dict)

In [ ]:
import graph_tool.all as gt

In [ ]:
from scipy.io import mmread

matrix = mmread("data/ia-reality/ia-reality.mtx")
graph_g = gt.Graph(matrix, directed=False)
pos = gt.fruchterman_reingold_layout(graph_g)
gt.graph_draw(graph_g, pos=pos)

In [ ]:
import matplotlib.pyplot as plt

partitioned = gt.minimize_blockmodel_dl(graph_g)
partitioned.draw(pos=pos)
partitioned.draw(pos=pos, vertex_fill_color=partitioned.get_blocks(), output="images/graph_tool_phone_communities.png")

print(f"{partitioned.get_nonempty_B()} total communities.")  

In [ ]:
# Create our dictionary 
blocks = partitioned.get_blocks()
gt_node_dict = {}
for index, item in enumerate(blocks):
    gt_node_dict[index] = item

print(gt_node_dict)
print(len(set(gt_node_dict.values())))

#### Question 2c

In [ ]:
# NetworkX version
from scipy.io import mmread

matrix = mmread("data/ia-reality/ia-reality.mtx")
mit_call_network = nx.Graph(matrix)

nx_call_network_rand = nx.double_edge_swap(mit_call_network, nswap=500, max_tries=5000)

nx_communities_rand = nx.community.louvain_communities(nx_call_network_rand, seed=42)

nx_node_dict_rand = {
    node: community_id 
    for community_id, community_nodes in enumerate(nx_communities_rand) 
    for node in community_nodes
}

pos = nx.spring_layout(nx_call_network_rand)
colors_sns = plt.cm.tab20b(np.linspace(0,1,len(np.unique(list(nx_communities_rand)))))
colors_nodes = [colors_sns[nx_node_dict_rand[node]] for node in nx_call_network_rand.nodes()]

fig, ax = plt.subplots(1,1,figsize=(12,12),dpi=100)

nx.draw_networkx(nx_call_network_rand, pos=pos, node_color=colors_nodes, with_labels=False, ax=ax, node_size=15,width=0.4,)
plt.title("Cell Phone Communities - Nx Deg Preserving Randomization")

plt.savefig("images/nx_rand_cellphone_communities.png")

plt.show()

In [ ]:
# Get the number of communities
print(f"Community count: {len(nx_communities_rand)}")

In [ ]:
# Graph Tool Version
import pyintergraph

# Create GT graph from networkX graph
gt_call_network_rand = pyintergraph.nx2gt(nx_call_network_rand, labelname="node_id")

pos = gt.fruchterman_reingold_layout(gt_call_network_rand)
gt.graph_draw(graph_g, pos=pos)
rand_state = gt.minimize_blockmodel_dl(gt_call_network_rand)
rand_state.draw(pos=pos)
rand_state.draw(pos=pos, vertex_fill_color=rand_state.get_blocks(), output="images/rand_gt_phone_communities.png")

In [ ]:
print(f"{rand_state.get_nonempty_B()} total communities.")  

In [ ]:
# Create dictionary 
rand_blocks = rand_state.get_blocks()
print(len(rand_state.get_blocks()))
gt_node_dict_rand = {}
for index, item in enumerate(rand_blocks):
    gt_node_dict_rand[index] = item

print(gt_node_dict_rand)
print(len(set(gt_node_dict_rand.values())))

#### Question 2c

In [ ]:
gt_node_dict

In [ ]:
from collections import Counter

cnt = Counter(gt_node_dict.values())

In [ ]:
# Network X Original Graph
community_dist = list(cnt.values())
ordered = dict(cnt.most_common())

# get nice binned distributions 
deg_bins = np.logspace(0, np.log10(np.max(community_dist)+1), num=len(community_dist))
probs, _ = np.histogram(community_dist, bins=deg_bins, density=True)
final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

plt.loglog(final_bins, probs, marker='o',lw=0, c="indigo")
plt.ylabel(r"P(Community Size)", fontsize='medium')
plt.xlabel(r"Community Size", fontsize='medium')
plt.grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')    
plt.title(f'NetworkX', fontsize='large')   

plt.show()

In [ ]:
from collections import Counter

fig, ax = plt.subplots(2,2,figsize=(12,12),dpi=100, sharex=True, sharey=True,)

# Network X Original Graph
cnt = Counter(gt_node_dict.values())
community_dist = list(cnt.values())

# get nice binned distributions 
deg_bins = np.linspace(0, np.max(community_dist), num=len(community_dist))
probs, _ = np.histogram(community_dist, bins=deg_bins, density=True)
final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

ax[0,0].plot(final_bins, probs, marker='o',lw=0, c="indigo")
ax[0,0].set_ylabel(r"P(Community Size)", fontsize='medium')
ax[0,0].set_xlabel(r"Community Size", fontsize='medium')
ax[0,0].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')    
ax[0,0].set_title(f'NetworkX', fontsize='large')  


# Network X Random Graph
cnt = Counter(nx_node_dict_rand.values())
community_dist = list(cnt.values())

# get nice binned distributions 
deg_bins = np.linspace(0, np.max(community_dist), num=len(community_dist))
probs, _ = np.histogram(community_dist, bins=deg_bins, density=True)
final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

ax[0,1].plot(final_bins, probs, marker='o',lw=0, c="indigo")
ax[0,1].set_ylabel(r"P(Community Size)", fontsize='medium')
ax[0,1].set_xlabel(r"Community Size", fontsize='medium')
ax[0,1].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')    
ax[0,1].set_title(f'NetworkX - Random', fontsize='large')  

# Graph-tool Original Graph
cnt = Counter(gt_node_dict.values())
community_dist = list(cnt.values())

# get nice binned distributions 
deg_bins = np.linspace(0, np.max(community_dist), num=len(community_dist))
probs, _ = np.histogram(community_dist, bins=deg_bins, density=True)
final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

ax[1,0].plot(final_bins, probs, marker='o',lw=0, c="indigo")
ax[1,0].set_ylabel(r"P(Community Size)", fontsize='medium')
ax[1,0].set_xlabel(r"Community Size", fontsize='medium')
ax[1,0].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')    
ax[1,0].set_title(f'Graph-tool', fontsize='large')  
  

# Graph-tool Random Graph
cnt = Counter(gt_node_dict_rand.values())
community_dist = list(cnt.values())

# get nice binned distributions 
deg_bins = np.linspace(0, np.max(community_dist), num=len(community_dist))
probs, _ = np.histogram(community_dist, bins=deg_bins, density=True)
final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

ax[1,1].plot(final_bins, probs, marker='o',lw=0, c="indigo")
ax[1,1].set_ylabel(r"P(Community Size)", fontsize='medium')
ax[1,1].set_xlabel(r"Community Size", fontsize='medium')
ax[1,1].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')    
ax[1,1].set_title(f'Graph-tool', fontsize='large')  

fig.suptitle('Distribution of Community Sizes', fontsize=14, fontweight='bold', y=0.9)
plt.subplots_adjust(top=0.85) 


plt.savefig('images/community_dist_graphs_lin.png', bbox_inches='tight')
plt.show()

In [ ]:
from collections import Counter

fig, ax = plt.subplots(2,2,figsize=(12,12),dpi=100, sharex=True, sharey=True,)

# Network X Original Graph
cnt = Counter(gt_node_dict.values())
community_dist = list(cnt.values())

# get nice binned distributions 
deg_bins = np.logspace(0, np.log10(np.max(community_dist)+1), num=len(community_dist))
probs, _ = np.histogram(community_dist, bins=deg_bins, density=True)
final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

ax[0,0].loglog(final_bins, probs, marker='o',lw=0, c="indigo")
ax[0,0].set_ylabel(r"P(Community Size)", fontsize='medium')
ax[0,0].set_xlabel(r"Community Size", fontsize='medium')
ax[0,0].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')    
ax[0,0].set_title(f'NetworkX', fontsize='large')  


# Network X Random Graph
cnt = Counter(nx_node_dict_rand.values())
community_dist = list(cnt.values())

# get nice binned distributions 
deg_bins = np.logspace(0, np.log10(np.max(community_dist)+1), num=len(community_dist))
probs, _ = np.histogram(community_dist, bins=deg_bins, density=True)
final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

ax[0,1].loglog(final_bins, probs, marker='o',lw=0, c="indigo")
ax[0,1].set_ylabel(r"P(Community Size)", fontsize='medium')
ax[0,1].set_xlabel(r"Community Size", fontsize='medium')
ax[0,1].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')    
ax[0,1].set_title(f'NetworkX - Random', fontsize='large')  

# Graph-tool Original Graph
cnt = Counter(gt_node_dict.values())
community_dist = list(cnt.values())

# get nice binned distributions 
deg_bins = np.logspace(0, np.log10(np.max(community_dist)+1), num=len(community_dist))
probs, _ = np.histogram(community_dist, bins=deg_bins, density=True)
final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

ax[1,0].loglog(final_bins, probs, marker='o',lw=0, c="indigo")
ax[1,0].set_ylabel(r"P(Community Size)", fontsize='medium')
ax[1,0].set_xlabel(r"Community Size", fontsize='medium')
ax[1,0].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')    
ax[1,0].set_title(f'Graph-tool', fontsize='large')  
  

# Graph-tool Random Graph
cnt = Counter(gt_node_dict_rand.values())
community_dist = list(cnt.values())

# get nice binned distributions 
deg_bins = np.logspace(0, np.log10(np.max(community_dist)+1), num=len(community_dist))
probs, _ = np.histogram(community_dist, bins=deg_bins, density=True)
final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

ax[1,1].loglog(final_bins, probs, marker='o',lw=0, c="indigo")
ax[1,1].set_ylabel(r"P(Community Size)", fontsize='medium')
ax[1,1].set_xlabel(r"Community Size", fontsize='medium')
ax[1,1].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')    
ax[1,1].set_title(f'Graph-tool', fontsize='large')  

fig.suptitle('Distribution of Community Sizes (loglog)', fontsize=14, fontweight='bold', y=0.9)
plt.subplots_adjust(top=0.85) 


plt.savefig('images/community_dist_graphs.png', bbox_inches='tight')
plt.show()

## Question 3

#### Question 3b

##### Question 3b i.

In [ ]:
import random

def get_rand_num():
    """
    Helper function to return a random float that doesn't include 0

    @returns float between (0,1)
    """
    num = random.random()
    while num == 0:
        num = random.random()
    return num

In [ ]:
import math
def get_power_law_seq(n, gamma, avg_deg):
    """
    This function creates a a sequence in a power-law distribution.

    @param n (int): The number of elements in the sequence
    @param gamma (float): The exponent of the powerlaw
    @param avg_deg (int): The average degree for the distribution

    @return a sequence of ints that form a power-law distribution
    """
    r_i = []
    # generate random numbers
    for i in range(0,n):
        rand = get_rand_num()
        r_i.append(rand)

    # Get k-min and k-max from average degree
    k_min = math.ceil(avg_deg * ((gamma-2)/(gamma-1)))
    k_max = math.ceil((k_min * n**(1/(gamma-1))))

    # Now get our probabilities for each possible degree,
    # based on the power-law distribution
    probs = []
    for k in range(k_min, k_max+1):
        # Get D(K)
        D_k = (k/k_min)**(-(gamma-1))
        probs.append(D_k)
        #print(f"K {k} is {D_k}")

    # Now use those probabilities to calculate degree count for each number
    final_degrees = []
    for r in r_i:
        for j,_ in enumerate(probs):
            # descend down the distribution until we are not smaller
            # than the next number 
            if j+1 >= len(probs) or r > probs[j+1]:
                final_degrees.append(j+k_min)
                break 
    final_degrees.sort()
    return final_degrees

In [ ]:
final_degrees = get_power_law_seq(10000,2.5,3)

In [ ]:
# get nice binned distributions 
deg_vals = final_degrees
deg_bins = np.logspace(0, np.log10(np.max(deg_vals)+1), num=40)
probs, _ = np.histogram(deg_vals, bins=deg_bins, density=True)
final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

# plot that puppy
plt.loglog(final_bins, probs, marker='o',lw=0)
plt.xlabel(r"$k$", fontsize='medium')
plt.ylabel(r"$P(k)$", fontsize='medium')
plt.grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
plt.title(f'Powerlaw Degree Distribution', fontsize='large')

plt.savefig('images/deg_dist_graphs.pdf', bbox_inches='tight')
plt.show()

##### Question 3b ii.

In [ ]:
def get_int_fraction(mu, degree):
    """
        This function gets the number of a node's degrees
        that connect to nodes in its own community.
    
        @param mu (float): The fraction of degrees external to a node's community
        @param gamma (degree): The number of degrees a node has
    
        @return (int) the number of degrees that must be internal
    """
    internal_degs = degree*(1-mu)
    num = random.random()

    # If between two degrees, we want the amount that
    # it returns either of the two degrees propotional
    # to the fraction
    if (num < internal_degs%round(internal_degs)):
        final_degs = round(internal_degs) + 1
    else:
        final_degs = round(internal_degs)
    return final_degs

In [ ]:
# Usage of the function 
final_degrees = get_power_law_seq(10000,3,3)
mu = .4

internal_deg_list = []
for node, degree in enumerate(final_degrees):
    # get the number of internal degrees 
    int_degs = get_int_fraction(mu, degree)
    internal_deg_list.append(int_degs)

print(f"Max internal degrees: {max(internal_deg_list)}")
print(f"Max degrees: {max(final_degrees)}")

# looks right-ish

#### Question 3c

In [ ]:
import time 

n_tracker = {}
gamma_tracker = {}
k_tracker = {}
mu_tracker = {}

for i_n in range(2,5):
    for i_gamma in range(0,10,2):
        for avg_degree in range(2,6):
            for i_mu in range(2,8,2):
                n = 10**i_n
                gamma = 2+i_gamma/10
                mu = i_mu / 10
                print(f"Testing with {n} nodes, gamma of {gamma}, mu of {mu} and average degree of {avg_degree}")
                start_time = time.perf_counter()
                try:
                    graph = nx.LFR_benchmark_graph(n, gamma, 1.5, mu=mu, average_degree=avg_degree)
                    end_time = time.perf_counter()
                    print(f"That ONE took {end_time - start_time} seconds to run. \n")
                    n_tracker[n] = n_tracker.get(n, 0) + 1
                    gamma_tracker[gamma] = gamma_tracker.get(gamma, 0) + 1
                    k_tracker[avg_degree] = k_tracker.get(avg_degree, 0) + 1
                    mu_tracker[mu] = mu_tracker.get(mu, 0) + 1
                except nx.ExceededMaxIterations:
                    print("Failed to converge, sad. \n")
                except Exception as e:
                    print(f"Other exception {e}")

            

In [ ]:
# Look closer at gamma
for i_gamma in range(0,10,2):
    for avg_degree in range(1,6,2):
        for i_mu in range(2,8,2):
            n = 10000
            gamma = 2+i_gamma/10
            mu = i_mu / 10
            print(f"Testing with {n} nodes, gamma of {gamma}, mu of {mu} and average degree of {avg_degree}")
            start_time = time.perf_counter()
            try:
                graph = nx.LFR_benchmark_graph(n, gamma, 1.5, mu=mu, average_degree=avg_degree)
                end_time = time.perf_counter()
                print(f"That one took {end_time - start_time} seconds to run. \n")
            except(nx.ExceededMaxIterations):
                print("Failed to converge. \n")

In [ ]:
# Look closer at average degree
for avg_degree in range(1,6):
    for i_mu in range(2,8,2):
        n = 10000
        gamma = 2.6
        mu = i_mu / 10
        print(f"Testing with {n} nodes, gamma of {gamma}, mu of {mu} and average degree of {avg_degree}")
        start_time = time.perf_counter()
        try:
            graph = nx.LFR_benchmark_graph(n, gamma, 1.5, mu=mu, average_degree=avg_degree)
            end_time = time.perf_counter()
            print(f"That one took {end_time - start_time} seconds to run. \n")
        except(nx.ExceededMaxIterations):
            print("Failed to converge. \n")

In [ ]:
n_tracker

In [ ]:
# Now look at the trackers

# n_tracker
# gamma_tracker
# k_tracker 
# mu_tracker 

fig, ax = plt.subplots(2,2,figsize=(10,10),dpi=100)

# N   
ax[0,0].bar([str(k) for k in n_tracker], list(n_tracker.values()), width=0.3)
ax[0,0].set_xlabel(r"Node counts", fontsize='medium')
ax[0,0].set_ylabel(r"Convergence Amount", fontsize='medium')
ax[0,0].set_title(f'N Convergence Rates', fontsize='large')
ax[0,0].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')

# Gamma   
ax[0,1].bar(gamma_tracker.keys(), gamma_tracker.values(), width=0.1)
ax[0,1].set_xlabel(r"Gamma values", fontsize='medium')
ax[0,1].set_ylabel(r"Convergence Amount", fontsize='medium')
ax[0,1].set_title(f'Gamma Convergence Rates', fontsize='large')
ax[0,1].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')

# Average Degree  
ax[1,0].bar(k_tracker.keys(), k_tracker.values(), width=0.1)
ax[1,0].set_xlabel(r"Average Degree values", fontsize='medium')
ax[1,0].set_ylabel(r"Convergence Amount", fontsize='medium')
ax[1,0].set_title(f'Average Degree Convergence Rates', fontsize='large')
ax[1,0].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')

# Mu   
ax[1,1].bar(mu_tracker.keys(), mu_tracker.values(), width=0.05)
ax[1,1].set_xlabel(r"Mu values", fontsize='medium')
ax[1,1].set_ylabel(r"Convergence Amount", fontsize='medium')
ax[1,1].set_title(f'Mu Convergence Rates', fontsize='large')
ax[1,1].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')



plt.savefig('images/model_convergence.png', bbox_inches='tight')
plt.show()

In [ ]:
import math
def get_power_law_seq(n, gamma, avg_deg):
    """
    This function creates a a sequence in a power-law distribution.

    @param n (int): The number of elements in the sequence
    @param gamma (float): The exponent of the powerlaw
    @param avg_deg (int): The average degree for the distribution

    @return a sequence of ints that form a power-law distribution
    """
    r_i = []
    # generate random numbers
    for i in range(0,n):
        rand = get_rand_num()
        r_i.append(rand)

    # Get k-min and k-max from average degree
    k_min = math.ceil(avg_deg * ((gamma-2)/(gamma-1)))
    k_max = math.ceil((k_min * n**(1/(gamma-1))))

    # Now get our probabilities for each possible degree,
    # based on the power-law distribution
    probs = []
    for k in range(k_min, k_max+1):
        # Get D(K)
        D_k = (k/k_min)**(-(gamma-1))
        probs.append(D_k)
        #print(f"K {k} is {D_k}")

    # Now use those probabilities to calculate degree count for each number
    final_degrees = []
    for r in r_i:
        for j,_ in enumerate(probs):
            # descend down the distribution until we are not smaller
            # than the next number 
            if j+1 >= len(probs) or r > probs[j+1]:
                final_degrees.append(j+k_min)
                break

    # must return an even number for the configuration model
    if sum(final_degrees) % 2 == 1:
        final_degrees[random.randint(0, len(final_degrees)-1)] += 1
        
    final_degrees.sort()
    return final_degrees

#### Question 3b iv.

In [ ]:
def get_community_sizes(n, gamma, avg_deg, beta):
    """
    This function creates a sizes of communities in a power-law distribution.

    @param n (int): The number of nodes in the network
    @param gamma (float): The exponent of the powerlaw of the degree distribution
    @param avg_deg (int): The average degree for the degree distribution
    @param beta (float): The exponent of the powerlaw of the community size distribution

    @return a sequence of ints that form a power-law distribution of community sizes
    """
    # Get k-min and k-max for the network
    k_min = math.ceil(avg_deg * ((gamma-2)/(gamma-1)))
    k_max = math.ceil((k_min * n**(1/(gamma-1))))

    # Now get the community sizes min and max
    s_min = k_min + 1
    s_max = k_max + 1

    # Now get our probabilities for every possible degree
    probs = []
    for s in range(s_min, s_max+1):
        # Get D(s)
        D_s = (s/s_min)**(-(beta-1))
        probs.append(D_s)
        #print(f"S {s} is {D_s}")

    # now get vals until we have total
    size_vals = []
    count = 0
    while count < n:
        r = get_rand_num()
        # descend down the distribution till we find our number
        for j,_ in enumerate(probs):
            # descend down the distribution until we are not smaller
            # than the next number 
            if j+1 >= len(probs) or r > probs[j+1]:
                size_vals.append(j+s_min)
                count = count + j + s_min
                break
        # now get rid of the extra nodes we can't fit in the last community
    size_vals[-1] = size_vals[-1] - (count - n)
    size_vals.sort()
    return size_vals

In [ ]:
# Implementation of the Entire Algorithm! 

N = 1000 # number of nodes
mu = .4
gamma = 2.5
beta = 1.5
avg_deg = 3
# get degree sequence 
final_degrees = get_power_law_seq(N,gamma,avg_deg)

# create network
G = nx.configuration_model(final_degrees)

for i in range(len(list(G.nodes()))):
    G.nodes[i]["degree"] = final_degrees[i]

    # get the node's internal degrees
    int_deg = get_int_fraction(mu, final_degrees[i])
    G.nodes[i]["int_deg"] = int_deg

    G.nodes[i]["homeless"] = True

# get our communities 
community_sizes = get_community_sizes(N, gamma, avg_deg, beta)

community_index = {i: [] for i in range(0,len(community_sizes))}

# everybody is homeless!
is_homeless = [True for _ in range(N)]

kill_me_count = 0
while True:
    if kill_me_count > 1000000:
        print("Alas, broken by the kill me count :( No convergence!")
        break
    # We done yet?
    if True not in is_homeless:
        break
    for node in G.nodes:
        # see if node has home
        if is_homeless[node]:
            random_community = random.randint(0, len(community_sizes)-1)
            if(community_sizes[random_community] > G.nodes[node]["int_deg"]):
                #print(f"Node {node} will be fit into community {random_community}")
                # is the community full? kick someone out
                if len(community_index[random_community]) >= community_sizes[random_community]:
                    # get random node in it
                    goneer = community_index[random_community][random.randint(0,len(community_index[random_community])-1)]
                    #print(f"But we are kicking out {community_index[random_community][goneer]} from community list {community_index[random_community]}")

                    # remove all forms of him
                    community_index[random_community].remove(goneer)
                    is_homeless[goneer] = True
                    G.nodes[goneer]["community"] = -1

                # now add the new guy 
                G.nodes[node]["community"] = random_community
                is_homeless[node] = False
                community_index[random_community].append(node)
    kill_me_count = kill_me_count + 1

# rewire all links
for node in G.nodes:
    internal_friends = []
    external_friends = []
    node_community = G.nodes[node]["community"]
    for edge in G.edges(node):
        friend = edge[1]
        if friend in community_index[node_community]:
            internal_friends.append(edge)
        else:
            external_friends.append(edge)
    friend_gap = G.nodes[node]["int_deg"] - len(internal_friends)
    for i in range(abs(friend_gap)):
        if friend_gap > 0 and len(external_friends) > 0:
            # we just do our best. if we don't have enough external friends, oh well
            # we want more internal friends!
            bye_friend = external_friends[random.randint(0, len(external_friends) - 1)]
            G.remove_edge(node, bye_friend[1])
            external_friends.remove(bye_friend)
            new_friend = community_index[node_community][random.randint(0, len(community_index[node_community]) - 2)] # minus two so we don't make friends with ourselves
            G.add_edge(node, new_friend)
        elif friend_gap < 0 and len(internal_friends) > 0:
            # too many internal friends :( 
            bye_friend = internal_friends[random.randint(0, len(internal_friends) - 1)]
            G.remove_edge(node, bye_friend[1])
            internal_friends.remove(bye_friend)
            new_friend = random.randint(0, N-1)
            while new_friend in community_index[node_community]:
                new_friend = random.randint(0, N-1)
            G.add_edge(node, new_friend)

                    

    
print(community_index)


In [ ]:
import numpy as np

# Now let's see how louvain communities do and visualize it

communities = nx.community.louvain_communities(G, seed=42)

nx_node_dict = {
    node: community_id 
    for community_id, community_nodes in enumerate(communities) 
    for node in community_nodes
}

pos = nx.spring_layout(G)
colors_sns = plt.cm.tab20b(np.linspace(0,1,len(np.unique(list(communities)))))
colors_nodes = [colors_sns[nx_node_dict[node]] for node in G.nodes()]

fig, ax = plt.subplots(1,1,figsize=(12,12),dpi=100)

nx.draw_networkx(G, pos=pos, node_color=colors_nodes, with_labels=False, ax=ax, node_size=15,width=0.4,)
plt.title("Benchmark Network Communities")

plt.show()

In [ ]:
G.degree()

In [ ]:
# And is it still a powerlaw?

# get nice binned distributions 
deg_vals = list(dict(G.degree()).values())
print(deg_vals)
deg_bins = np.logspace(0, np.log10(np.max(deg_vals)+1), num=40)
probs, _ = np.histogram(deg_vals, bins=deg_bins, density=True)
final_bins = deg_bins[1:] - np.diff(deg_bins)/2.0

# plot that puppy
plt.loglog(final_bins, probs, marker='o',lw=0)
plt.xlabel(r"$k$", fontsize='medium')
plt.ylabel(r"$P(k)$", fontsize='medium')
plt.grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
plt.title(f'Benchmark Network Degree Distribution', fontsize='large')

plt.savefig('images/deg_dist_graphs.pdf', bbox_inches='tight')
plt.show()

## Question 4

The package can be installed with `pip install -i https://test.pypi.org/simple/ gilbert-nets==0.0.5`

In [ ]:
from gilbert_nets import networkfx

networkfx.test_all()